In [ ]:
pip install langchain langchain-google-genai faiss-cpu pypdf

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader("/Users/kamal/Desktop/AI-Projects/REFRAG/The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf")
pages = loader.load()

# Extract text
full_text = " ".join([page.page_content for page in pages])

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(full_text)

print(f"Total chunks: {len(chunks)}")


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
chunk_embeddings = embeddings.embed_documents(chunks)

In [ ]:
import faiss
import numpy as np

dim = len(chunk_embeddings[0])
index = faiss.IndexFlatL2(dim)
index.add(np.array(chunk_embeddings).astype("float32"))

def retrieve_chunks(query, top_k=5):
    query_vec = embeddings.embed_query(query)
    distances, indices = index.search(np.array([query_vec], dtype="float32"), top_k)
    retrieved = [chunks[i] for i in indices[0]]
    return retrieved, distances[0]


In [ ]:
def selective_expand(query, top_k=5, threshold=0.5):
    retrieved_chunks, distances = retrieve_chunks(query, top_k)
    expanded = []
    compressed = []

    for chunk, dist in zip(retrieved_chunks, distances):
        sim = 1 / (1 + dist)  
        if sim > threshold:
            expanded.append(chunk)  
        else:
            compressed.append(chunk[:200] + "...")  

    return expanded, compressed


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

In [ ]:
query = "Summarize the key points about Breast Cancer."

expanded, compressed = selective_expand(query)

context = "\n\n".join(expanded + compressed)

messages = [
    HumanMessage(content=f"Answer the following using the context below:\n{context}\n\nQuestion: {query}")
]

In [ ]:
response = llm.invoke(messages)
print(response.content)